# HuggingFace Dataset Viewer

Generic interactive viewer for any HuggingFace dataset. Configure the cell below and run all.

In [ ]:
# --- Configuration ---
DATASET_ID = "isaacus/legal-rag-bench"
QA_CONFIG = "qa"
CORPUS_CONFIG = "corpus"
SPLIT = "test"

# Display settings
TEXT_COLUMNS = ["question", "answer", "passage_text"]  # Large text blocks
TAG_COLUMNS = ["id", "relevant_passage_id"]  # Coloured tags
SORT_COLUMNS = ["question", "passage_text"]  # Sort by text length

# Optional post-merge filter function (set to None for no filtering)
FILTER_FN = None

In [ ]:
from datasets import load_dataset

qa_ds = load_dataset(DATASET_ID, name=QA_CONFIG, split=SPLIT)
corpus_ds = load_dataset(DATASET_ID, name=CORPUS_CONFIG, split=SPLIT)

# Build an id -> text lookup from corpus and attach passage text to QA rows
corpus_text_by_id = {row["id"]: row["text"] for row in corpus_ds}
ds = qa_ds.map(lambda ex: {"passage_text": corpus_text_by_id.get(ex["relevant_passage_id"])})

if FILTER_FN:
    ds = ds.filter(FILTER_FN)

print(f"Dataset: {DATASET_ID} | qa + corpus ({SPLIT})")
print(f"Rows: {len(ds):,}")
print(f"Columns: {ds.column_names}")

In [ ]:
import html as _html

import ipywidgets as widgets
from IPython.display import HTML, display

# --- Colours for tags and text blocks ---
_TAG_COLOURS = [
    "#e8eaf6",
    "#e0f2f1",
    "#fff3e0",
    "#fce4ec",
    "#f3e5f5",
    "#e1f5fe",
    "#fff9c4",
    "#dcedc8",
]
_TEXT_COLOURS = [
    ("#1565c0", "#f5f7ff"),  # blue
    ("#2e7d32", "#f5fdf5"),  # green
    ("#6a1b9a", "#faf5ff"),  # purple
    ("#e65100", "#fff8f5"),  # orange
    ("#00838f", "#f5fdfd"),  # teal
]

# --- Data preparation ---
df_viewer = ds.to_pandas()

# Add length columns for text columns
for col in TEXT_COLUMNS:
    if col in df_viewer.columns:
        df_viewer[f"_len_{col}"] = df_viewer[col].astype(str).str.len()

# Sort by text column lengths if requested
if SORT_COLUMNS:
    sort_keys = [f"_len_{c}" for c in SORT_COLUMNS if f"_len_{c}" in df_viewer.columns]
    if sort_keys:
        df_viewer = df_viewer.sort_values(sort_keys).reset_index(drop=True)

total = len(df_viewer)


# --- Build viewer ---
def _make_tag(value, colour):
    return (
        f'<span style="background:{colour}; padding:2px 8px; '
        f'border-radius:10px; margin-right:6px;">'
        f"{_html.escape(str(value))}</span>"
    )


def _make_text_block(title, text, accent, bg):
    return (
        f'<div style="border-left:4px solid {accent}; background:{bg};'
        f' padding:12px 16px; margin-bottom:12px; border-radius:0 4px 4px 0;">'
        f'<div style="font-weight:600; color:{accent}; margin-bottom:6px; font-size:14px;">'
        f"{_html.escape(title)}</div>"
        f'<div style="white-space:pre-wrap; font-size:14px; line-height:1.5;">'
        f"{_html.escape(str(text))}</div></div>"
    )


btn_prev = widgets.Button(description="\u25c0 Prev", layout=widgets.Layout(width="100px"))
btn_next = widgets.Button(description="Next \u25b6", layout=widgets.Layout(width="100px"))
counter = widgets.HTML(value="")
out = widgets.Output()
state = [0]


def render(idx):
    row = df_viewer.iloc[idx]
    counter.value = (
        f'<span style="font-family:monospace; font-size:14px;">Example {idx + 1} / {total}</span>'
    )

    # Tags
    tags_html = ""
    tag_cols = [c for c in TAG_COLUMNS if c in df_viewer.columns]
    if tag_cols:
        tags = "".join(
            _make_tag(row[c], _TAG_COLOURS[i % len(_TAG_COLOURS)]) for i, c in enumerate(tag_cols)
        )
        tags_html = f'<div style="margin-bottom:8px; color:#555; font-size:13px;">{tags}</div>'

    # Text blocks
    text_blocks = ""
    for i, col in enumerate(TEXT_COLUMNS):
        if col not in df_viewer.columns:
            continue
        accent, bg = _TEXT_COLOURS[i % len(_TEXT_COLOURS)]
        length = row.get(f"_len_{col}", len(str(row[col])))
        text_blocks += _make_text_block(f"{col} ({length} chars)", row[col], accent, bg)

    card = (
        f'<div style="font-family:system-ui,sans-serif; max-width:900px;">'
        f"{tags_html}{text_blocks}</div>"
    )
    out.clear_output(wait=True)
    with out:
        display(HTML(card))


def on_prev(b):
    if state[0] > 0:
        state[0] -= 1
        render(state[0])


def on_next(b):
    if state[0] < total - 1:
        state[0] += 1
        render(state[0])


btn_prev.on_click(on_prev)
btn_next.on_click(on_next)

nav_bar = widgets.HBox(
    [btn_prev, counter, btn_next],
    layout=widgets.Layout(align_items="center", gap="12px", margin="0 0 8px 0"),
)
viewer = widgets.VBox([nav_bar, out])

render(0)
display(viewer)